<a href="https://colab.research.google.com/github/paulmunozpauta/Curso_Introduccion_Ciencia_Datos/blob/main/Notebooks/M04_01_Correlacion_causalidad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<p align="center">
  <img src="https://github.com/paulmunozpauta/Curso_Introduccion_Ciencia_Datos/raw/main/Static/Imgs/UdeC_color_horizontal.jpg" width="500">
</p>

<p align="center"><b style="font-size:28px;">Facultad de Ingeniería Agrícola</b></p>
<p align="center"><b style="font-size:28px;">Introducción a la Ciencia de Datos</b></p>
<hr>
<p align="center"><b style="font-size:28px;">Contacto</b></p>
<p align="center">
  paulmunoz@udec.cl<br>
  https://paulmunoz.com
</p>

# Introducción a la Ciencia de Datos

## Módulo 4: Análisis e interpretación de datos

### Notebook 1: Correlación y causalidad

En los módulos anteriores aprendimos a importar, organizar, limpiar y explorar datos. También utilizamos herramientas de estadística descriptiva y visualización para comprender el comportamiento de variables ambientales.

Hasta ahora, nuestros análisis se han centrado principalmente en estudiar las características de una variable individual.

Sin embargo, en ingeniería agrícola y ambiental frecuentemente necesitamos comprender **cómo se relacionan diferentes variables**.

Por ejemplo:

- ¿Existe una relación entre la temperatura del aire y la humedad relativa?
- ¿Cómo cambia la temperatura cuando aumenta la radiación solar?
- ¿La presión atmosférica está relacionada con otras variables meteorológicas?
- ¿Una relación estadística significa necesariamente que una variable causa cambios en otra?

Para responder estas preguntas utilizaremos datos reales de la estación meteorológica de bajo costo **MeteoChillán**, ubicada en el Campus Chillán de la Universidad de Concepción.

En este notebook aprenderemos a explorar y cuantificar relaciones entre variables, además de reconocer las limitaciones de su interpretación.

Al finalizar este notebook podrás:

- Identificar relaciones entre variables mediante diagramas de dispersión.
- Distinguir relaciones positivas, negativas y no lineales.
- Calcular e interpretar coeficientes de correlación de Pearson y Spearman.
- Construir e interpretar matrices de correlación.
- Evaluar cómo los valores extremos pueden influir en una correlación.
- Comprender por qué **correlación no implica causalidad**.

## 1. Importar las librerías

Utilizaremos las bibliotecas que ya conocemos:

- `pandas`: para organizar y analizar los datos.
- `matplotlib.pyplot`: para configurar y personalizar figuras.
- `seaborn`: para construir visualizaciones estadísticas.

Mantendremos un formato gráfico consistente para facilitar la comparación e interpretación de los resultados.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

plt.rcParams.update({
    "font.size": 16,
    "axes.titlesize": 18,
    "axes.labelsize": 16,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14
})

## 2. Cargar los datos de MeteoChillán

Utilizaremos los datos de la estación meteorológica de bajo costo **MeteoChillán**, ubicada en el Campus Chillán de la Universidad de Concepción.

Esta estación registra diferentes variables meteorológicas, entre ellas:

- Temperatura del aire (°C).
- Humedad relativa (%).
- Presión atmosférica (hPa).
- Velocidad del viento (m/s).
- Radiación solar (W/m²).
- Precipitación (mm).

En este notebook analizaremos cómo se relacionan algunas de estas variables.

Para ello, necesitamos que las observaciones correspondan a los **mismos instantes de tiempo**. No tendría sentido comparar la temperatura de las 10:00 con la humedad relativa de las 15:00 como si fueran observaciones simultáneas.

Utilizaremos el archivo de MeteoChillán empleado anteriormente en el Módulo 2.

**Fuente:** [MeteoChillán — Universidad de Concepción](https://meteochillan.udec.cl/stations.html)

In [2]:
from google.colab import files

uploaded = files.upload()

nombre_archivo = next(iter(uploaded))
datos = pd.read_csv(nombre_archivo)

print("Dimensiones:", datos.shape)
print("\nColumnas disponibles:")
print(datos.columns.tolist())

datos.head()

Saving AW005_historical_5min.csv to AW005_historical_5min.csv
Dimensiones: (33253, 27)

Columnas disponibles:
['datetime_utc', 'air_temperature_c', 'relative_humidity_pct', 'dew_point_c', 'feels_like_c', 'app_temperature_c', 'vpd', 'indoor_temperature_c', 'indoor_relative_humidity_pct', 'indoor_dew_point_c', 'indoor_feels_like_c', 'solar_radiation_w_m2', 'uv_index', 'rain_rate_mm_hr', 'hourly_rain_mm', 'daily_rain_mm', 'event_rain_mm', 'weekly_rain_mm', 'monthly_rain_mm', 'yearly_rain_mm', 'wind_direction_deg', 'relative_pressure', 'absolute_pressure', 'wind_speed_ms', 'wind_gust_ms', 'absolute_pressure_hpa', 'relative_pressure_hpa']


,datetime_utc,air_temperature_c,relative_humidity_pct,dew_point_c,feels_like_c,app_temperature_c,vpd,indoor_temperature_c,indoor_relative_humidity_pct,indoor_dew_point_c,...,weekly_rain_mm,monthly_rain_mm,yearly_rain_mm,wind_direction_deg,relative_pressure,absolute_pressure,wind_speed_ms,wind_gust_ms,absolute_pressure_hpa,relative_pressure_hpa
0,2025-12-22 05:00:00+00:00,19.2,57,10.5,19.2,19.4,0.283,18.8,54,9.3,...,0.0,4.3,4.3,202,29.93,29.46,0.0,0.0,997.630494,1013.546527
1,2025-12-22 05:30:00+00:00,18.9,58,10.4,18.9,19.1,0.274,18.6,55,9.3,...,0.0,4.3,4.3,202,29.93,29.46,0.0,0.0,997.630494,1013.546527
2,2025-12-22 06:00:00+00:00,18.7,58,10.3,18.7,18.8,0.267,18.3,55,9.2,...,0.0,4.3,4.3,202,29.92,29.45,0.0,0.0,997.291855,1013.207888
3,2025-12-22 06:30:00+00:00,18.5,59,10.2,18.5,18.6,0.259,18.1,56,9.2,...,0.0,4.3,4.3,202,29.90,29.43,0.0,0.0,996.614577,1012.530610
4,2025-12-22 07:00:00+00:00,18.3,59,10.1,18.3,18.3,0.254,17.9,56,9.0,...,0.0,4.3,4.3,202,29.90,29.44,0.0,0.0,996.953216,1012.530610
